### Run this notebook online

[![Open in Colab](https://img.shields.io/badge/Open_in-Colab-F9AB00?logo=googlecolab&logoColor=F9AB00)](https://colab.research.google.com/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/12_Naive_Sequential_Reference.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https%3A%2F%2Fgithub.com%2Fhosein-fanai%2FContinual-Learning-with-Diffusion-Vision-Transformers%2Fblob%2Fmain%2Fnotebooks%2Fthesis%2F12_Naive_Sequential_Reference.ipynb)
[![Launch Binder](https://img.shields.io/badge/launch-binder-F5793A?logo=jupyter&logoColor=white)](https://mybinder.org/v2/gh/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/main?urlpath=lab%2Ftree%2Fnotebooks%2Fthesis%2F12_Naive_Sequential_Reference.ipynb)

- **Google Colab:** open the notebook, select a GPU for training under **Runtime > Change runtime type**, then choose **Run all**.
- **Kaggle:** sign in and import the notebook, enable **Internet**, select a **GPU** accelerator for training, then **Run all**.
- **Binder:** opens a temporary CPU JupyterLab session. Use it to inspect the notebook or run small checks; full training needs more resources.
- **[Studio Lab](https://studiolab.sagemaker.aws/import/github/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/12_Naive_Sequential_Reference.ipynb) (existing accounts only):** start a runtime, copy the notebook to your project, select a Python **3.11–3.13** kernel, and set `RUNTIME = "studiolab"` in the first code cell before **Run all**. For CPU, also set `CUDA = False`.

The **first code cell** finds or downloads the repository and prepares TensorFlow **2.20** / Keras **3.11.2** before project imports. If setup requests a restart, restart the kernel and run all again. For another hosted Jupyter service, set `RUNTIME = "hosted"` (`CUDA = False` for CPU or compatible provider-managed CUDA). Locally, select the project TensorFlow kernel.

Launch links open the published GitHub `main` version; publish this notebook and its setup files together before using them. For a notebook that has not been published, upload its `.ipynb` file to Colab or Kaggle instead. GPU availability depends on the provider. Save checkpoints and results before a temporary session ends.

Benchmark training and collection notebooks require the campaign artifacts prepared in notebook **01**. This supplemental reference runs independently.

See the [hosted runtime guide](https://github.com/hosein-fanai/Continual-Learning-with-Diffusion-Vision-Transformers/blob/main/notebooks/thesis/README.md#hosted-runtimes) for setup and import details.


In [ ]:
# Shared setup: use the local initializer when available, otherwise download it.
from pathlib import Path
from urllib.request import urlopen


CHECKOUT_NAME = "Continual-Learning-with-Diffusion-Vision-Transformers"
REPOSITORY = f"https://github.com/hosein-fanai/{CHECKOUT_NAME}.git"
REVISION = "main"
RUNTIME = "auto"  # Use "hosted" for another online service, or "local" to verify only.
CUDA = None  # False: CPU or managed CUDA; True: retain CUDA pip dependencies.

_locations = (Path.cwd(), *Path.cwd().parents, Path.cwd() / CHECKOUT_NAME,
              Path("/kaggle/working") / CHECKOUT_NAME, Path("/content") / CHECKOUT_NAME)
_initializer = next((path / "notebooks" / "init.py" for path in _locations
                     if (path / "notebooks" / "init.py").is_file()), None)
_url = f"https://raw.githubusercontent.com/hosein-fanai/{CHECKOUT_NAME}/{REVISION}/notebooks/init.py"
_setup = {"__name__": "notebook_setup", "__file__": str(_initializer or _url)}
with (_initializer.open("rb") if _initializer else urlopen(_url, timeout=30)) as _file:
    exec(compile(_file.read(), _setup["__file__"], "exec"), _setup)
ROOT, RUNTIME_PACKAGES = _setup["prepare_notebook"](
    checkout_name=CHECKOUT_NAME, repository=REPOSITORY, revision=REVISION,
    runtime=RUNTIME, cuda=CUDA,
)


# 12 - Naive sequential fine-tuning reference

Train one model through the scheduled tasks using only each current task's training rows. This measures retention when no continual-learning intervention is applied.

Select **TensorFlow 2.20 (Docker GPU)**, restart the kernel, then run top to bottom. These notebooks create supplemental reference results; no measured results are included yet.

## 1. Check the runtime

The maintained environment is TensorFlow 2.20 with Keras 3. Run this notebook after other GPU training has finished.

In [ ]:
import os


os.environ.setdefault("TF_FORCE_GPU_ALLOW_GROWTH", "true")

from IPython.display import display
from notebooks.thesis.workflow import check_runtime


print(check_runtime())
from notebooks.thesis.reference_benchmarks import (
    configure_reference, prepare_reference, train_reference, finish_reference,
)

## 2. Select a paired experiment

The defaults remain **CIFAR-10, seed 17, validation evaluation**. Choose CIFAR-100 in the same cell to run its reference. These defaults do not register a separate fixed reference campaign or automatically pair this run with the benchmark seeds. For a comparison, match the seed, class order, training partition, evaluation split and scoring policy.

Use `test` only after fixing the supplemental reference recipe and scope. Such a run remains outside the 24-stream benchmark and its collector. The known prior test exposure from historical HPO still applies; a fixed recipe and fresh seed do not create independent confirmation.

In [ ]:
DATASET = "cifar10"  # "cifar10" or "cifar100"
SEED = 17  # Match the comparison run's seed and class order.
EVALUATION_SPLIT = "validation"  # Set "test" only after fixing the recipe.

In [ ]:
BENCHMARK = 'naive_sequential'
config = configure_reference(
    DATASET, BENCHMARK, seed=SEED, evaluation_split=EVALUATION_SPLIT,
)

## 3. Review the training plan

The native `joint_none` incremental path carries the model forward between tasks. Gradients use only current-task training rows: no old-data replay, stored-example buffer, knowledge distillation, or semantic route. Evaluation on earlier held-out tasks measures retention without training on them.

Both references retain the thesis DiT architecture and combined diffusion/classification objective. The current recipe uses Adam at initial LR **0.001**, zero weight decay, **50 epochs/task** and batch size **128**. Evaluation uses raw primary-head predictions only (`clf_acc_coef=1.0`, `clf_distil_acc_coef=0.0`), averaged uniformly across **64 retained timesteps** from `max_t=256`, `t_range_drop_rate=0.75`. Inverse-SNR weighting selects timesteps for removal; it does not weight the retained predictions (`weighted=false`).

During `prepare_reference`, the cosine duration is resolved from each task's actual permitted training rows before optimizer creation, replacing the inherited replay-platform horizon. With the default uncapped 20% validation split, CIFAR-10 uses `5 × 50 × ceil(8000 / 128) = 15,750` updates and CIFAR-100 uses `10 × 50 × ceil(4000 / 128) = 16,000`. Every task retains its final partial batch each epoch. This is one whole-stream cosine, not a per-task restart. The prepared configuration and recorded `training_budget` are authoritative when splits, schedules or data limits change. Equal nominal epochs per permitted example do not establish equal compute with replay methods.

In [ ]:
schedule = config.continually_learn
display({
    "dataset": config.dataset.name,
    "reference": BENCHMARK,
    "seed": schedule.seed,
    "evaluation_split": EVALUATION_SPLIT,
    "epochs_per_training_example": config.training.epochs,
    "batch_size": config.dataset.batch_size,
    "class_order": schedule.class_order,
    "task_groups": schedule.task_groups,
})

## 4. Prepare data, model, and result folder

The helper applies the selected seed, class schedule and held-out split, then resolves the reference's actual cosine duration before creating its optimizer. Do not change the configuration after preparation.

In [ ]:
context = prepare_reference(config)
print("Results folder:", context["run_dir"])

## 5. Train once

This is the full reference run and may take substantial time. A repeated training-cell execution is rejected for the same prepared context. For a fresh experiment, restart the kernel and run from the beginning.

In [ ]:
# The helper refuses a second training call for the same prepared context.
history = train_reference(config, context)

## 6. Save and inspect results

Inspect final overall accuracy, the per-task evaluation table, and available forgetting/backward-transfer results from the task sequence. Poor retention is an observation to measure, not an assumed result.

In [ ]:
summary, per_task = finish_reference(config, context, history)
display(summary)
display(per_task)
print("Saved results:", context["run_dir"])

## Interpretation and cleanup

Treat this result as a naive sequential lower reference. It is not the minimum possible accuracy, and a continual-learning intervention can perform worse. Compare identical dataset splits, seeds, class orders, task groups, and evaluation conventions.

After the final cell succeeds, **save this notebook with its outputs, then restart its kernel** to release its training state before running another notebook. Result artifacts are saved separately in the printed folder.

See the [reference definitions and comparison limits](README.md#offline-and-naive-reference-benchmarks).